# 04: Simple Data Pipeline

In this notebook, you'll build an end-to-end data pipeline: fetch → clean → transform → save.

## Learning Objectives

- Build a complete data pipeline with separate stages
- Create synthetic transaction data to simulate real-world scenarios
- Apply cleaning and transformation in sequence
- Save cleaned data for downstream use
- Structure the pipeline as a reusable Python script

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta

print(f"Pandas version: {pd.__version__}")
print(f"Pipeline started at: {datetime.now()}")

## 2. Fetch Stage: Create Synthetic Transaction Data

In production, you'd fetch data from an API or database. Here we simulate realistic transaction data with intentional quality issues.

In [ ]:
np.random.seed(42)

# Generate 200 synthetic transactions with realistic issues
n = 200

# Random dates over the past year
start_date = datetime(2024, 1, 1)
dates = [start_date + timedelta(days=np.random.randint(0, 365)) for _ in range(n)]

# Transaction amounts (skewed distribution)
amounts = np.random.lognormal(mean=3.5, sigma=1.0, size=n).round(2)

# Categories
categories = np.random.choice(
    ["Electronics", "Clothing", "Food", "Home", "Sports", "Books"],
    size=n, p=[0.25, 0.20, 0.20, 0.15, 0.10, 0.10]
)

# Customer IDs
customer_ids = np.random.randint(1000, 1050, size=n)

# Payment methods
payment_methods = np.random.choice(
    ["credit_card", "debit_card", "paypal", "bank_transfer"],
    size=n, p=[0.4, 0.3, 0.2, 0.1]
)

df_raw = pd.DataFrame({
    "transaction_id": [f"TXN-{10000+i}" for i in range(n)],
    "date": dates,
    "customer_id": customer_ids,
    "category": categories,
    "amount": amounts,
    "payment_method": payment_methods
})

# --- Inject data quality issues ---

# 1. Add missing values (about 8% of rows)
missing_idx = np.random.choice(n, size=int(n * 0.08), replace=False)
df_raw.loc[missing_idx[:10], "amount"] = np.nan
df_raw.loc[missing_idx[10:15], "category"] = None

# 2. Add duplicates (5 duplicate rows)
dup_rows = df_raw.sample(5, random_state=42)
df_raw = pd.concat([df_raw, dup_rows], ignore_index=True)

# 3. Add inconsistent category names
df_raw.loc[10, "category"] = "electronics"  # lowercase
df_raw.loc[20, "category"] = " CLOTHING "  # whitespace
df_raw.loc[30, "category"] = "FOOD"  # all caps

# 4. Add invalid amounts
df_raw.loc[50, "amount"] = -100.0  # negative
df_raw.loc[60, "amount"] = 999999.99  # extreme outlier

# 5. Add inconsistent payment method names
df_raw.loc[70, "payment_method"] = "Credit Card"  # different format
df_raw.loc[80, "payment_method"] = "credit card"  # space instead of underscore

# 6. Convert date to string (simulate messy format)
df_raw["date"] = df_raw["date"].apply(
    lambda d: d.strftime("%Y-%m-%d") if pd.notna(d) else d
)

print(f"Raw data created: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
print(f"\nIssues injected:")
print(f"  Missing values: {df_raw.isnull().sum().sum()}")
print(f"  Duplicate rows: {df_raw.duplicated().sum()}")
print(f"  Negative amounts: {(df_raw['amount'] < 0).sum()}")

df_raw.head(10)

## 3. Clean Stage

In [ ]:
def clean_transactions(df):
    """Clean raw transaction data."""
    df = df.copy()
    
    print(f"[CLEAN] Starting with {len(df)} rows")
    
    # 1. Remove duplicates
    n_before = len(df)
    df = df.drop_duplicates()
    print(f"[CLEAN] Removed {n_before - len(df)} duplicate rows")
    
    # 2. Standardize category names
    df["category"] = df["category"].str.strip().str.title()
    
    # 3. Standardize payment method names
    payment_map = {
        "credit card": "credit_card",
        "Credit Card": "credit_card",
        "debit card": "debit_card",
        "Debit Card": "debit_card",
        "bank transfer": "bank_transfer",
        "Bank Transfer": "bank_transfer"
    }
    df["payment_method"] = df["payment_method"].str.strip().str.lower().str.replace(" ", "_")
    
    # 4. Handle invalid amounts
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
    df.loc[df["amount"] <= 0, "amount"] = np.nan
    
    # 5. Drop rows with missing critical fields
    df = df.dropna(subset=["transaction_id", "customer_id"])
    
    # 6. Fill missing amounts with median per category
    df["amount"] = df.groupby("category")["amount"].transform(
        lambda x: x.fillna(x.median())
    )
    
    # 7. Fill missing categories with mode
    df["category"] = df["category"].fillna(df["category"].mode()[0])
    
    # 8. Convert date column
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    
    # 9. Drop rows with invalid dates
    df = df.dropna(subset=["date"])
    
    # 10. Reset index
    df = df.reset_index(drop=True)
    
    print(f"[CLEAN] Finished with {len(df)} rows")
    print(f"[CLEAN] Remaining nulls: {df.isnull().sum().sum()}")
    
    return df

df_cleaned = clean_transactions(df_raw)

In [ ]:
# Verify cleaning results
print("=== After Cleaning ===")
print(f"Rows: {len(df_cleaned)}")
print(f"Missing values: {df_cleaned.isnull().sum().sum()}")
print(f"Duplicates: {df_cleaned.duplicated().sum()}")
print(f"\nCategories: {df_cleaned['category'].unique()}")
print(f"Payment methods: {df_cleaned['payment_method'].unique()}")

df_cleaned.head()

## 4. Transform Stage

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

def transform_transactions(df):
    """Transform cleaned transaction data for analysis/ML."""
    df = df.copy()
    
    print(f"[TRANSFORM] Starting with {len(df)} rows, {len(df.columns)} columns")
    
    # 1. Date features
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day_of_week"] = df["date"].dt.dayofweek
    df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
    
    # 2. Amount features
    scaler = StandardScaler()
    df["amount_scaled"] = scaler.fit_transform(df[["amount"]])
    
    # Amount bins
    df["amount_tier"] = pd.cut(
        df["amount"],
        bins=[0, 20, 50, 150, float("inf")],
        labels=["small", "medium", "large", "very_large"]
    )
    
    # 3. One-hot encode category
    df = pd.get_dummies(df, columns=["category"], prefix="cat", drop_first=True)
    
    # 4. One-hot encode payment method
    df = pd.get_dummies(df, columns=["payment_method"], prefix="pay", drop_first=True)
    
    # 5. Customer-level features (aggregation)
    customer_stats = df.groupby("customer_id").agg(
        total_spent=("amount", "sum"),
        avg_transaction=("amount", "mean"),
        transaction_count=("transaction_id", "count")
    ).reset_index()
    
    df = df.merge(customer_stats, on="customer_id", how="left")
    
    print(f"[TRANSFORM] Finished with {len(df)} rows, {len(df.columns)} columns")
    
    return df

df_transformed = transform_transactions(df_cleaned)

In [ ]:
# Final dataset overview
print("=== Final Dataset ===")
print(f"Shape: {df_transformed.shape}")
print(f"\nColumn types:")
print(df_transformed.dtypes)

df_transformed.head()

## 5. Save Stage

In [ ]:
def save_pipeline_output(df, output_dir="data"):
    """Save pipeline output files."""
    os.makedirs(output_dir, exist_ok=True)
    
    # Save main output
    output_path = os.path.join(output_dir, "transactions_transformed.csv")
    df.to_csv(output_path, index=False)
    print(f"[SAVE] Saved {len(df)} rows to {output_path}")
    
    # Save summary stats
    summary = {
        "total_rows": len(df),
        "total_columns": len(df.columns),
        "total_customers": df["customer_id"].nunique(),
        "date_range": f"{df['date'].min()} to {df['date'].max()}",
        "total_revenue": round(df["amount"].sum(), 2)
    }
    
    summary_df = pd.DataFrame([summary])
    summary_path = os.path.join(output_dir, "pipeline_summary.csv")
    summary_df.to_csv(summary_path, index=False)
    print(f"[SAVE] Summary saved to {summary_path}")
    
    return summary

summary = save_pipeline_output(df_transformed)

In [ ]:
# Print final summary
print("\n" + "="*50)
print("PIPELINE COMPLETE")
print("="*50)
for key, value in summary.items():
    print(f"  {key}: {value}")

## 6. Pipeline as a Script

The notebook functions above can be combined into a standalone Python script. Here's the structure:

In [ ]:
# This is what pipeline.py would look like:

pipeline_script = """
import pandas as pd
import numpy as np
import os
from datetime import datetime
from sklearn.preprocessing import StandardScaler


def fetch_data(filepath):
    """Load raw data from CSV."""
    print(f"Loading data from {filepath}")
    return pd.read_csv(filepath)


def clean_data(df):
    """Clean raw data."""
    df = df.copy()
    df = df.drop_duplicates()
    df["category"] = df["category"].str.strip().str.title()
    df["payment_method"] = df["payment_method"].str.strip().str.lower().str.replace(" ", "_")
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
    df.loc[df["amount"] <= 0, "amount"] = np.nan
    df["amount"] = df.groupby("category")["amount"].transform(lambda x: x.fillna(x.median()))
    df["category"] = df["category"].fillna(df["category"].mode()[0])
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date", "transaction_id", "customer_id"])
    df = df.reset_index(drop=True)
    print(f"Cleaned: {len(df)} rows")
    return df


def transform_data(df):
    """Transform data for ML/analysis."""
    df = df.copy()
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day_of_week"] = df["date"].dt.dayofweek
    df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
    scaler = StandardScaler()
    df["amount_scaled"] = scaler.fit_transform(df[["amount"]])
    df["amount_tier"] = pd.cut(df["amount"], bins=[0, 20, 50, 150, float("inf")],
                               labels=["small", "medium", "large", "very_large"])
    df = pd.get_dummies(df, columns=["category"], prefix="cat", drop_first=True)
    df = pd.get_dummies(df, columns=["payment_method"], prefix="pay", drop_first=True)
    customer_stats = df.groupby("customer_id").agg(
        total_spent=("amount", "sum"),
        avg_transaction=("amount", "mean"),
        transaction_count=("transaction_id", "count")
    ).reset_index()
    df = df.merge(customer_stats, on="customer_id", how="left")
    print(f"Transformed: {len(df)} rows, {len(df.columns)} columns")
    return df


def save_data(df, output_path):
    """Save transformed data."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df.to_csv(output_path, index=False)
    print(f"Saved to {output_path}")


def main():
    print(f"Pipeline started at {datetime.now()}")
    raw = fetch_data("data/transactions_raw.csv")
    cleaned = clean_data(raw)
    transformed = transform_data(cleaned)
    save_data(transformed, "data/transactions_transformed.csv")
    print(f"Pipeline completed at {datetime.now()}")


if __name__ == "__main__":
    main()
"""

print("Pipeline script structure:")
print("=" * 40)
print(pipeline_script)

## 7. Pipeline Metrics

Track data quality metrics at each stage.

In [ ]:
def pipeline_report(raw, cleaned, transformed):
    """Generate a pipeline quality report."""
    report = pd.DataFrame({
        "metric": [
            "rows", "columns", "missing_values", "duplicates",
            "numeric_columns", "categorical_columns"
        ],
        "raw": [
            len(raw), len(raw.columns), raw.isnull().sum().sum(),
            raw.duplicated().sum(),
            len(raw.select_dtypes(include=[np.number]).columns),
            len(raw.select_dtypes(include=["object"]).columns)
        ],
        "cleaned": [
            len(cleaned), len(cleaned.columns), cleaned.isnull().sum().sum(),
            cleaned.duplicated().sum(),
            len(cleaned.select_dtypes(include=[np.number]).columns),
            len(cleaned.select_dtypes(include=["object"]).columns)
        ],
        "transformed": [
            len(transformed), len(transformed.columns), transformed.isnull().sum().sum(),
            transformed.duplicated().sum(),
            len(transformed.select_dtypes(include=[np.number]).columns),
            len(transformed.select_dtypes(include=["object"]).columns)
        ]
    })
    
    print("\n=== Pipeline Quality Report ===")
    print(report.to_string(index=False))
    return report

report = pipeline_report(df_raw, df_cleaned, df_transformed)

## Exercises

### Exercise 1: Extend the pipeline

Add a new stage to the pipeline that:
1. Calculates monthly revenue by category
2. Identifies the top 5 customers by total spend
3. Saves these as separate CSV files

### Exercise 2: Add error handling

Modify the pipeline functions to:
1. Handle file-not-found errors gracefully
2. Validate that required columns exist before processing
3. Log errors with timestamps

### Exercise 3: Build a different pipeline

Create a pipeline for a different domain:
1. Generate synthetic student grades data (name, subject, score, date)
2. Inject quality issues (missing scores, duplicate entries, invalid dates)
3. Build clean → transform → save pipeline
4. Add a feature: letter grade (A/B/C/D/F) based on score

### Bonus: Schedule the pipeline

If you have experience with cron or task schedulers:
1. Write the pipeline as `pipeline.py`
2. Create a cron job or simple script that runs it daily
3. Save output with a date stamp: `data/transactions_2024-01-15.csv`

In [ ]:
# Your code here for Exercise 1
# ...